# Módulo 2 — Desarrollador Full Stack de Soluciones Inteligentes

## Unidad I: Redes Neuronales y Aplicaciones Full Stack

### Clase 3: Creacion de API del proyecto (FastAPI) para que el frontend pueda consumir predicciones

---



### Actualización del archivo `requirements.txt` con las dependencias de la API

tensorflow
pandas
numpy
scikit-learn
joblib
matplotlib
openml
fastapi
uvicorn[standard]

### Creación del modulo `src/api.py`

In [ ]:
# API de predicción de ingresos (>50K / <=50K) - Versión corta para clase de 1h
import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES

# Dónde están los archivos guardados al entrenar
PREPROCESSOR_PATH = "artifacts/preprocessor.joblib"
MODEL_PATH = "artifacts/model.keras"
preprocessor = None
model = None


def load_artifacts():
    global preprocessor, model
    if not os.path.isfile(PREPROCESSOR_PATH) or not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError("Falta entrenar: python -m src.train")
    preprocessor = joblib.load(PREPROCESSOR_PATH)
    model = tf.keras.models.load_model(MODEL_PATH)


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: cargar modelo y preprocesador antes de aceptar peticiones
    try:
        load_artifacts()
    except FileNotFoundError as e:
        print("Aviso:", e)
    yield
    # Shutdown: (opcional) liberar recursos si hiciera falta


app = FastAPI(title="Income ML API", version="1.0", lifespan=lifespan)

app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:3000", "http://localhost:5173"], allow_methods=["*"], allow_headers=["*"])


# Formato del JSON que debe enviar el frontend en POST /predict
class PredictRequest(BaseModel):
    age: int
    fnlwgt: int
    education_num: int
    capital_gain: int
    capital_loss: int
    hours_per_week: int
    workclass: str
    education: str
    marital_status: str
    occupation: str
    relationship: str
    race: str
    sex: str
    native_country: str


@app.get("/")
def root():
    """Página principal: estado y enlace a documentación."""
    return {"message": "Income ML API", "docs": "/docs", "model_loaded": model is not None}


@app.get("/features")
def get_features():
    """Lista de variables del modelo (el frontend la usa para el formulario)."""
    if preprocessor is None:
        try:
            load_artifacts()
        except FileNotFoundError:
            raise HTTPException(503, "Entrena antes: python -m src.train")
    opts = {}
    cat = preprocessor.named_transformers_.get("cat")
    if cat and hasattr(cat.named_steps.get("encoder"), "categories_"):
        for name, arr in zip(CATEGORICAL_FEATURES, cat.named_steps["encoder"].categories_):
            opts[name] = [str(x) for x in arr]
    return {"numeric": NUMERIC_FEATURES, "categorical": list(CATEGORICAL_FEATURES), "categorical_options": opts}


@app.post("/predict")
def predict(body: PredictRequest):
    """Recibe datos de una persona; devuelve predicción >50K o <=50K y probabilidad."""
    if preprocessor is None or model is None:
        try:
            load_artifacts()
        except FileNotFoundError:
            raise HTTPException(503, "Modelo no cargado. Ejecuta: python -m src.train")

    row = {
        "age": body.age, "fnlwgt": body.fnlwgt, "education-num": body.education_num,
        "capital-gain": body.capital_gain, "capital-loss": body.capital_loss,
        "hours-per-week": body.hours_per_week,
        "workclass": body.workclass, "education": body.education,
        "marital-status": body.marital_status, "occupation": body.occupation,
        "relationship": body.relationship, "race": body.race, "sex": body.sex,
        "native-country": body.native_country,
    }
    X = preprocessor.transform(pd.DataFrame([row]))
    if hasattr(X, "toarray"):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    prob = float(model.predict(X, verbose=0)[0][0])
    return {"prediction": ">50K" if prob > 0.5 else "<=50K", "probability": round(prob, 4)}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run("src.api:app", host="127.0.0.1", port=8000, reload=True)


## Creación del Front con React

1. Instalar Node.js versión LTS

    `https://nodejs.org/en`

2. Verificar la instalación

    `node -v`

    `npm -v`

3. En la carpeta `INCOME-ML-SYSTEM ejecutar

    `npm create vite@latest income-ml-frontend -- --template react`

4. Cambiar la carpeta de cache de npm desde PowerShell

    `npm config set cache "C:\Users\luisg\npm-cache"`

5. Instala las dependencias del frontend

    `npm install`

6. Instalar dependencias del proyecto

    `npm install react react-dom`
    `npm install -D @vitejs/plugin-react @types/react @types/react-dom`

5. Cambiar directorio

    `cd income-ml-frontend`

6. Levantar el Front

    `npm run dev`

## Creacion de componentes

`income-ml-frontend/app/lib/api.ts`

In [ ]:
/**
 * URL base de la API de predicción (backend FastAPI).
 * En desarrollo: el frontend corre en localhost:5173 y la API en localhost:8000.
 */
export const API_BASE = "http://localhost:8000";

export interface PredictRequest {
  age: number;
  fnlwgt: number;
  education_num: number;
  capital_gain: number;
  capital_loss: number;
  hours_per_week: number;
  workclass: string;
  education: string;
  marital_status: string;
  occupation: string;
  relationship: string;
  race: string;
  sex: string;
  native_country: string;
}

export interface PredictResponse {
  prediction: ">50K" | "<=50K";
  probability: number;
}

export interface FeaturesResponse {
  numeric: string[];
  categorical: string[];
  categorical_options: Record<string, string[]>;
}


`income-ml-frontend/app/lib/options.ts`

In [ ]:
/**
 * Opciones por defecto para variables categóricas del dataset Adult (OpenML 1590).
 * Se usan cuando la API aún no tiene el modelo entrenado y no devuelve /features.
 */
export const DEFAULT_CATEGORICAL_OPTIONS: Record<string, string[]> = {
  workclass: [
    "Private",
    "Self-emp-not-inc",
    "Self-emp-inc",
    "Federal-gov",
    "Local-gov",
    "State-gov",
    "Without-pay",
    "Never-worked",
  ],
  education: [
    "Bachelors",
    "Some-college",
    "11th",
    "HS-grad",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc",
    "9th",
    "7th-8th",
    "12th",
    "Masters",
    "1st-4th",
    "10th",
    "Doctorate",
    "5th-6th",
    "Preschool",
  ],
  "marital-status": [
    "Married-civ-spouse",
    "Divorced",
    "Never-married",
    "Separated",
    "Widowed",
    "Married-spouse-absent",
    "Married-AF-spouse",
  ],
  occupation: [
    "Tech-support",
    "Craft-repair",
    "Other-service",
    "Sales",
    "Exec-managerial",
    "Prof-specialty",
    "Handlers-cleaners",
    "Machine-op-inspct",
    "Adm-clerical",
    "Farming-fishing",
    "Transport-moving",
    "Priv-house-serv",
    "Protective-serv",
    "Armed-Forces",
  ],
  relationship: [
    "Wife",
    "Own-child",
    "Husband",
    "Not-in-family",
    "Other-relative",
    "Unmarried",
  ],
  race: [
    "White",
    "Asian-Pac-Islander",
    "Amer-Indian-Eskimo",
    "Other",
    "Black",
  ],
  sex: ["Female", "Male"],
  "native-country": [
    "United-States",
    "Cambodia",
    "England",
    "Puerto-Rico",
    "Canada",
    "Germany",
    "India",
    "Japan",
    "Greece",
    "China",
    "Cuba",
    "Iran",
    "Honduras",
    "Philippines",
    "Italy",
    "Poland",
    "Jamaica",
    "Vietnam",
    "Mexico",
    "Portugal",
    "Ireland",
    "France",
    "Dominican-Republic",
    "Laos",
    "Ecuador",
    "Taiwan",
    "Haiti",
    "Columbia",
    "Hungary",
    "Guatemala",
    "Nicaragua",
    "Scotland",
    "Thailand",
    "Yugoslavia",
    "El-Salvador",
    "Trinadad&Tobago",
    "Peru",
    "Hong",
    "Holand-Netherlands",
  ],
};


`income-ml-frontend/app/components/PredictForm.tsx`

In [ ]:
import { useState, useEffect } from "react";
import { API_BASE } from "../lib/api";
import type { PredictRequest, PredictResponse, FeaturesResponse } from "../lib/api";
import { DEFAULT_CATEGORICAL_OPTIONS } from "../lib/options";

const initialForm: PredictRequest = {
  age: 35,
  fnlwgt: 77516,
  education_num: 10,
  capital_gain: 0,
  capital_loss: 0,
  hours_per_week: 40,
  workclass: "Private",
  education: "Bachelors",
  marital_status: "Never-married",
  occupation: "Adm-clerical",
  relationship: "Not-in-family",
  race: "White",
  sex: "Male",
  native_country: "United-States",
};

export function PredictForm() {
  const [form, setForm] = useState<PredictRequest>(initialForm);
  const [options, setOptions] = useState<Record<string, string[]>>(DEFAULT_CATEGORICAL_OPTIONS);
  const [result, setResult] = useState<PredictResponse | null>(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState<string | null>(null);
  const [apiStatus, setApiStatus] = useState<"checking" | "ok" | "error">("checking");

  // Cargar opciones categóricas desde la API (si el modelo está entrenado)
  useEffect(() => {
    fetch(`${API_BASE}/features`)
      .then((res) => {
        if (res.ok) return res.json() as Promise<FeaturesResponse>;
        throw new Error("API no disponible");
      })
      .then((data) => {
        if (data.categorical_options && Object.keys(data.categorical_options).length > 0) {
          setOptions(data.categorical_options);
        }
        setApiStatus("ok");
      })
      .catch(() => setApiStatus("error"));
  }, []);

  const handleChange = (field: keyof PredictRequest, value: string | number) => {
    setForm((prev) => ({ ...prev, [field]: value }));
    setResult(null);
    setError(null);
  };

  const handleSubmit = async (e: React.FormEvent) => {
    e.preventDefault();
    setLoading(true);
    setError(null);
    setResult(null);
    try {
      const res = await fetch(`${API_BASE}/predict`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(form),
      });
      const data = await res.json();
      if (!res.ok) throw new Error(data.detail || "Error en la predicción");
      setResult(data);
    } catch (err) {
      setError(err instanceof Error ? err.message : "Error de conexión");
    } finally {
      setLoading(false);
    }
  };

  const categoricalKeys = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country",
  ] as const;

  // Mapeo: nombre en el formulario -> clave en options (API usa "marital-status", form usa "marital_status")
  const optionKey: Record<string, string> = {
    workclass: "workclass",
    education: "education",
    marital_status: "marital-status",
    occupation: "occupation",
    relationship: "relationship",
    race: "race",
    sex: "sex",
    native_country: "native-country",
  };

  return (
    <main className="min-h-screen bg-slate-50 dark:bg-slate-900 text-slate-800 dark:text-slate-200">
      <div className="max-w-2xl mx-auto px-4 py-8">
        <header className="mb-8 text-center">
          <h1 className="text-2xl font-bold text-slate-900 dark:text-white">
            Predicción de ingresos (Adult Dataset)
          </h1>
          <p className="text-slate-600 dark:text-slate-400 mt-1">
            Introduce los datos y el modelo predecirá si los ingresos son &gt;50K o ≤50K.
          </p>
          {apiStatus === "checking" && (
            <p className="text-amber-600 mt-2 text-sm">Comprobando conexión con la API…</p>
          )}
          {apiStatus === "error" && (
            <p className="text-amber-600 mt-2 text-sm">
              La API no está disponible. Asegúrate de ejecutar el backend: <code className="bg-slate-200 dark:bg-slate-700 px-1 rounded">python -m src.api</code>
            </p>
          )}
        </header>

        <form onSubmit={handleSubmit} className="space-y-6 bg-white dark:bg-slate-800 rounded-xl shadow-lg p-6">
          {/* Campos numéricos */}
          <section>
            <h2 className="text-lg font-semibold mb-3 text-slate-700 dark:text-slate-300">Datos numéricos</h2>
            <div className="grid grid-cols-1 sm:grid-cols-2 gap-4">
              <label className="block">
                <span className="block text-sm font-medium mb-1">Edad (age)</span>
                <input
                  type="number"
                  min={1}
                  max={120}
                  value={form.age}
                  onChange={(e) => handleChange("age", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Peso final (fnlwgt)</span>
                <input
                  type="number"
                  min={0}
                  value={form.fnlwgt}
                  onChange={(e) => handleChange("fnlwgt", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Años de educación (education-num)</span>
                <input
                  type="number"
                  min={1}
                  max={16}
                  value={form.education_num}
                  onChange={(e) => handleChange("education_num", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Ganancia de capital (capital-gain)</span>
                <input
                  type="number"
                  min={0}
                  value={form.capital_gain}
                  onChange={(e) => handleChange("capital_gain", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Pérdida de capital (capital-loss)</span>
                <input
                  type="number"
                  min={0}
                  value={form.capital_loss}
                  onChange={(e) => handleChange("capital_loss", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Horas por semana (hours-per-week)</span>
                <input
                  type="number"
                  min={1}
                  max={99}
                  value={form.hours_per_week}
                  onChange={(e) => handleChange("hours_per_week", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
            </div>
          </section>

          {/* Campos categóricos */}
          <section>
            <h2 className="text-lg font-semibold mb-3 text-slate-700 dark:text-slate-300">Datos categóricos</h2>
            <div className="grid grid-cols-1 sm:grid-cols-2 gap-4">
              {categoricalKeys.map((key) => {
                const opts = options[optionKey[key]] || [];
                const label = key.replace(/_/g, " ");
                return (
                  <label key={key} className="block">
                    <span className="block text-sm font-medium mb-1">{label}</span>
                    <select
                      value={form[key]}
                      onChange={(e) => handleChange(key, e.target.value)}
                      className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                    >
                      {opts.map((opt) => (
                        <option key={opt} value={opt}>{opt}</option>
                      ))}
                    </select>
                  </label>
                );
              })}
            </div>
          </section>

          <div className="flex flex-col sm:flex-row gap-3 pt-2">
            <button
              type="submit"
              disabled={loading}
              className="px-4 py-2 bg-blue-600 text-white rounded-lg font-medium hover:bg-blue-700 disabled:opacity-50 disabled:cursor-not-allowed"
            >
              {loading ? "Prediciendo…" : "Predecir ingresos"}
            </button>
            <button
              type="button"
              onClick={() => { setForm(initialForm); setResult(null); setError(null); }}
              className="px-4 py-2 border border-slate-300 dark:border-slate-600 rounded-lg font-medium hover:bg-slate-100 dark:hover:bg-slate-700"
            >
              Restablecer valores
            </button>
          </div>

          {error && (
            <div className="p-3 rounded-lg bg-red-100 dark:bg-red-900/30 text-red-800 dark:text-red-200 text-sm">
              {error}
            </div>
          )}

          {result && (
            <div className="p-4 rounded-lg bg-slate-100 dark:bg-slate-700 border border-slate-200 dark:border-slate-600">
              <h3 className="font-semibold mb-2">Resultado de la predicción</h3>
              <p className="text-lg">
                Ingresos previstos: <strong>{result.prediction}</strong>
              </p>
              <p className="text-sm text-slate-600 dark:text-slate-400 mt-1">
                Probabilidad de &gt;50K: {(result.probability * 100).toFixed(2)}%
              </p>
            </div>
          )}
        </form>
      </div>
    </main>
  );
}


`income-ml-frontend/app/routes/home.tsx`

In [ ]:
import type { Route } from "./+types/home";
import { PredictForm } from "../components/PredictForm";

export function meta({}: Route.MetaArgs) {
  return [
    { title: "Predicción de ingresos - Income ML" },
    { name: "description", content: "Aplicación de predicción de ingresos (>50K / ≤50K) con modelo de aprendizaje automático." },
  ];
}

export default function Home() {
  return <PredictForm />;
}


### Cómo ejecutarlo

En la raiz del proyecto

    `pip install -r requirements.txt`

    `python -m src.train`

    `python -m src.api`

Del lado del Front

    `cd income-ml-frontend`

    `npm install`

    `npm run dev`